In [1]:
# Load repo-local path configuration.
import os
from pathlib import Path

from pa3.utils.env import load_dotenv

load_dotenv()
PA3_REPO_ROOT = Path(os.environ["PA3_REPO_ROOT"]).expanduser()


def pa3_path(relative_path: str) -> str:
    return str(PA3_REPO_ROOT / relative_path)


In [2]:
import json
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

ROOT = Path(pa3_path("Reproduce/results"))

# Or discover many files at once, e.g.:
# file_paths = sorted(WORKSPACE.glob("vmess_*_DF.json"))

ALL_PROTOCOLS = ("vmess", "shadowsocks", "trojan")
# Test order matches scripts/*_test.sh calls for each training protocol
TEST_PROTOCOL_ORDER = {
    "vmess": ("shadowsocks", "trojan"),
    "shadowsocks": ("vmess", "trojan"),
    "trojan": ("vmess", "shadowsocks"),
}

MODEL_FEATURE_MAP = {
    "DF": "size_bin",
    "BAPM": "size_bin",
    "TF": "size_bin",
    "NetCLR": "size_bin",
    "TikTok": "dt",
    "RF": "tsam"
}

MODELS = ["DF", "BAPM", "TF", "NetCLR", "TikTok", "RF"]

PROTOCOL_LABELS = {p: p.capitalize() for p in ALL_PROTOCOLS}

def parse_f1_scores(file_path: Path) -> list[float]:
    with open(file_path, "r") as f:
        content = f.read()
    json_objects = re.findall(r"\{.*?\}", content, re.DOTALL)
    f1_scores = []
    for obj_str in json_objects:
        try:
            data = json.loads(obj_str)
            score = data.get("F1-score")
            if score is not None:
                f1_scores.append(score)
        except json.JSONDecodeError:
            print(f"Invalid JSON object skipped in {file_path.name}.")
    return f1_scores


def infer_train_protocol(stem: str) -> Optional[str]:
    for protocol in ALL_PROTOCOLS:
        if stem == protocol or stem.startswith(f"{protocol}_"):
            return protocol
    return None


def format_mean_std(values, decimals: int = 3) -> str:
    return (
        f"{np.round(np.mean(values), decimals):.3f} "
        f"($\\pm {np.round(np.std(values), decimals):.3f}$)"
    )


def build_f1_table(file_paths, decimals: int = 3) -> pd.DataFrame:
    rows = []
    column_order = []

    for file_path in file_paths:
        file_path = Path(file_path)
        if not file_path.exists():
            continue
        stem = file_path.stem
        train_protocol = infer_train_protocol(stem)
        if train_protocol is None:
            raise ValueError(f"Cannot infer training protocol from filename: {stem}")

        test_protocols = TEST_PROTOCOL_ORDER[train_protocol]
        f1_scores = parse_f1_scores(file_path)
        n_protocols = len(test_protocols)
        if len(f1_scores) % n_protocols != 0:
            raise ValueError(
                f"{stem}: expected F1 count divisible by {n_protocols}, got {len(f1_scores)}"
            )

        row = {"file": stem.split("_")[-1]}
        for i, protocol in enumerate(test_protocols):
            label = PROTOCOL_LABELS[protocol]
            protocol_scores = f1_scores[i::n_protocols]
            row[label] = format_mean_std(protocol_scores, decimals)
            if label not in column_order:
                column_order.append(label)
        rows.append(row)

    df = pd.DataFrame(rows).set_index("file").T
    return df

def build_paths(train_protocol, method):

    files = []

    for model in MODELS:
        if method == "baseline":
            files.append(f"{train_protocol}_{MODEL_FEATURE_MAP[model]}_{model}")
        else:
            files.append(f"{train_protocol}_{MODEL_FEATURE_MAP[model]}_merge_{model}")

    file_paths = [
        ROOT / f"{method}/{file}.json" for file in files
    ]

    return file_paths


In [4]:
for train_protocol in ALL_PROTOCOLS:
    method = "pa3"
    file_paths = build_paths(train_protocol, method)
    df = build_f1_table(file_paths)
    print(f"Training protocol: {train_protocol}")
    print(f"{df.index[0]}: " + " & ".join(df.iloc[0].astype(str)))
    print(f"{df.index[1]}: " + " & ".join(df.iloc[1].astype(str)))

Training protocol: vmess
Shadowsocks: 0.774 ($\pm 0.008$) & 0.653 ($\pm 0.008$) & 0.966 ($\pm 0.001$) & 0.743 ($\pm 0.008$) & 0.927 ($\pm 0.003$) & 0.758 ($\pm 0.007$)
Trojan: 0.633 ($\pm 0.011$) & 0.582 ($\pm 0.005$) & 0.931 ($\pm 0.004$) & 0.572 ($\pm 0.009$) & 0.830 ($\pm 0.007$) & 0.686 ($\pm 0.012$)
Training protocol: shadowsocks
Vmess: 0.605 ($\pm 0.006$) & 0.497 ($\pm 0.006$) & 0.930 ($\pm 0.003$) & 0.524 ($\pm 0.009$) & 0.727 ($\pm 0.006$) & 0.580 ($\pm 0.018$)
Trojan: 0.600 ($\pm 0.010$) & 0.528 ($\pm 0.005$) & 0.905 ($\pm 0.001$) & 0.518 ($\pm 0.009$) & 0.783 ($\pm 0.006$) & 0.619 ($\pm 0.004$)
Training protocol: trojan
Vmess: 0.600 ($\pm 0.010$) & 0.530 ($\pm 0.013$) & 0.928 ($\pm 0.003$) & 0.538 ($\pm 0.016$) & 0.750 ($\pm 0.007$) & 0.636 ($\pm 0.011$)
Shadowsocks: 0.691 ($\pm 0.021$) & 0.523 ($\pm 0.019$) & 0.942 ($\pm 0.003$) & 0.576 ($\pm 0.030$) & 0.860 ($\pm 0.006$) & 0.737 ($\pm 0.006$)
